In [ ]:
!nvidia-smi

In [ ]:
# =========================
# CELL 1A — MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# CELL 1B — IMPORTS + CONFIG
# =========================
import os, json, copy, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# PATH CONFIG — PROJECT
# =========================
PROJECT_DIR = "/content/drive/MyDrive/S-Class/Orion/OrionFL/orion-fl-dr-research"

DATASET_NAME    = "MultiClient"
EXPERIMENT_NAME = "exp8_fl_dann_size_asc"
MODEL_NAME      = "mobilenet"

BASE_RESULT_DIR = os.path.join(PROJECT_DIR, "results", DATASET_NAME, EXPERIMENT_NAME, MODEL_NAME)
MODELS_DIR      = os.path.join(BASE_RESULT_DIR, "models")
LOGS_DIR        = os.path.join(BASE_RESULT_DIR, "logs")
FIGURES_DIR     = os.path.join(BASE_RESULT_DIR, "figures")

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(LOGS_DIR,    exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# =========================
# PATH CONFIG — CLIENTS
# =========================
DDR_NPZ_PATH       = "/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DDR_dataset_224.npz"
DDR_LABEL_PATH     = "/content/drive/MyDrive/S-Class/Orion/OrionFL/DDR_Dataset/DR_grading.csv"

EYEPACS_NPZ_PATH   = "/content/drive/MyDrive/S-Class/Orion/OrionFL/EyePACS_Dataset/training/EyePACS_dataset_224.npz"
EYEPACS_LABEL_PATH = "/content/drive/MyDrive/S-Class/Orion/OrionFL/EyePACS_Dataset/training/trainLabels.csv"

APTOS_NPZ_PATH     = "/content/drive/MyDrive/S-Class/Orion/OrionFL/APTOS_2019/preprocessed/orion_dr_224.npz"

# =========================
# FL + DANN CONFIG
# =========================
IMG_SIZE               = (224, 224)
BATCH_SIZE             = 32
NUM_CLASSES            = 5
NUM_DOMAINS            = 3          # DDR=0, EyePACS=1, APTOS=2
NUM_CLIENTS            = 3
CLIENT_NAMES           = ["DDR", "EyePACS", "APTOS"]

FL_ROUNDS              = 15
EPOCHS_LOCAL_STAGE1    = 3
EPOCHS_LOCAL_STAGE2    = 3

LEARNING_RATE_STAGE1   = 1e-3
LEARNING_RATE_STAGE2   = 1e-6
UNFREEZE_LAST_N_LAYERS = 10

# DANN hyperparameters
LAMBDA_GAMMA           = 10.0   # progressive lambda gamma (paper DANN original)
DOMAIN_LOSS_WEIGHT     = 0.2    # bobot domain loss dalam total loss
FEATURE_DIM            = 128    # dimensi shared feature space

SORT_STRATEGY          = "size_ascending"
SORTED_ORDER_FIXED    = [2, 0, 1]  # APTOS → DDR → EyePACS (smallest → largest)

# =========================
# DEBUG INFO
# =========================
print("TensorFlow version :", tf.__version__)
print("SORT_STRATEGY      :", SORT_STRATEGY)
print("FL_ROUNDS          :", FL_ROUNDS)
print("DOMAIN_LOSS_WEIGHT :", DOMAIN_LOSS_WEIGHT)
print("LAMBDA_GAMMA       :", LAMBDA_GAMMA)
print()
print("PROJECT_DIR exists     :", os.path.exists(PROJECT_DIR))
print("DDR_NPZ exists         :", os.path.exists(DDR_NPZ_PATH))
print("DDR_LABEL exists       :", os.path.exists(DDR_LABEL_PATH))
print("EYEPACS_NPZ exists     :", os.path.exists(EYEPACS_NPZ_PATH))
print("EYEPACS_LABEL exists   :", os.path.exists(EYEPACS_LABEL_PATH))
print("APTOS_NPZ exists       :", os.path.exists(APTOS_NPZ_PATH))
print("\nAll imports OK.")

In [ ]:
# =========================
# CELL 2 — LOAD CLIENT 0: DDR
# =========================
ddr_data     = np.load(DDR_NPZ_PATH, allow_pickle=True)
print("DDR NPZ keys:", ddr_data.files)
X_ddr        = ddr_data["X"]
ddr_label_df = pd.read_csv(DDR_LABEL_PATH)
y_ddr        = ddr_label_df["diagnosis"].values.astype(np.int64)

assert len(X_ddr) == len(y_ddr), f"DDR mismatch: {len(X_ddr)} vs {len(y_ddr)}"
print("DDR — X:", X_ddr.shape, "| y:", y_ddr.shape)
print(pd.Series(y_ddr).value_counts().sort_index())

In [ ]:
# =========================
# CELL 3 — LOAD CLIENT 1: EyePACS
# =========================
eyepacs_data     = np.load(EYEPACS_NPZ_PATH, allow_pickle=True)
X_eyepacs        = eyepacs_data["images"]
eyepacs_label_df = pd.read_csv(EYEPACS_LABEL_PATH)
y_eyepacs        = eyepacs_label_df["level"].values.astype(np.int64)

assert len(X_eyepacs) == len(y_eyepacs), f"EyePACS mismatch: {len(X_eyepacs)} vs {len(y_eyepacs)}"
print("EyePACS — X:", X_eyepacs.shape, "| y:", y_eyepacs.shape)
print(pd.Series(y_eyepacs).value_counts().sort_index())

In [ ]:
# =========================
# CELL 4 — LOAD CLIENT 2: APTOS
# =========================
aptos_data = np.load(APTOS_NPZ_PATH, allow_pickle=True)
X_aptos    = aptos_data["images"]
y_aptos    = aptos_data["labels"].astype(np.int64)

assert len(X_aptos) == len(y_aptos), f"APTOS mismatch: {len(X_aptos)} vs {len(y_aptos)}"
print("APTOS — X:", X_aptos.shape, "| y:", y_aptos.shape)
print(pd.Series(y_aptos).value_counts().sort_index())

In [ ]:
# =========================
# CELL 5 — PER-CLIENT TRAIN/VAL/TEST SPLIT (70:15:15)
# =========================
def split_client(X, y, seed=SEED):
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    X_v, X_te, y_v, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp
    )
    return X_tr, X_v, X_te, y_tr, y_v, y_te

X_tr_ddr, X_v_ddr, X_te_ddr, y_tr_ddr, y_v_ddr, y_te_ddr = split_client(X_ddr,     y_ddr)
X_tr_eye, X_v_eye, X_te_eye, y_tr_eye, y_v_eye, y_te_eye = split_client(X_eyepacs, y_eyepacs)
X_tr_apt, X_v_apt, X_te_apt, y_tr_apt, y_v_apt, y_te_apt = split_client(X_aptos,   y_aptos)

print(f"  DDR     — train: {len(y_tr_ddr):>6,} | val: {len(y_v_ddr):>5,} | test: {len(y_te_ddr):>5,}")
print(f"  EyePACS — train: {len(y_tr_eye):>6,} | val: {len(y_v_eye):>5,} | test: {len(y_te_eye):>5,}")
print(f"  APTOS   — train: {len(y_tr_apt):>6,} | val: {len(y_v_apt):>5,} | test: {len(y_te_apt):>5,}")

X_val_global  = np.concatenate([X_v_ddr,  X_v_eye,  X_v_apt],  axis=0)
y_val_global  = np.concatenate([y_v_ddr,  y_v_eye,  y_v_apt],  axis=0)
X_test_global = np.concatenate([X_te_ddr, X_te_eye, X_te_apt], axis=0)
y_test_global = np.concatenate([y_te_ddr, y_te_eye, y_te_apt], axis=0)

src_test_global = np.array(
    ["DDR"]     * len(y_te_ddr) +
    ["EyePACS"] * len(y_te_eye) +
    ["APTOS"]   * len(y_te_apt)
)

print(f"\nGlobal val  : {len(y_val_global):,}")
print(f"Global test : {len(y_test_global):,}")

In [ ]:
# =========================
# CELL 6 — PREPROCESS (MobileNetV2 preprocess_input)
# =========================
def preproc(X):
    return preprocess_input(X.astype("float32"))

X_tr_ddr = preproc(X_tr_ddr); X_v_ddr = preproc(X_v_ddr); X_te_ddr = preproc(X_te_ddr)
X_tr_eye = preproc(X_tr_eye); X_v_eye = preproc(X_v_eye); X_te_eye = preproc(X_te_eye)
X_tr_apt = preproc(X_tr_apt); X_v_apt = preproc(X_v_apt); X_te_apt = preproc(X_te_apt)

X_val_global  = preproc(X_val_global)
X_test_global = preproc(X_test_global)
print("Preprocessing done.")

In [ ]:
# =========================
# CELL 7 — BUILD CLIENT DATA STRUCTURE
# =========================
# Domain ID: DDR=0, EyePACS=1, APTOS=2 (sesuai index CLIENT_NAMES)
client_data = [
    (X_tr_ddr, y_tr_ddr),    # Client 0 — DDR      (domain_id=0)
    (X_tr_eye, y_tr_eye),    # Client 1 — EyePACS  (domain_id=1)
    (X_tr_apt, y_tr_apt),    # Client 2 — APTOS    (domain_id=2)
]

client_val_data = [
    (X_v_ddr, y_v_ddr),
    (X_v_eye, y_v_eye),
    (X_v_apt, y_v_apt),
]

client_test_data = [
    (X_te_ddr, y_te_ddr),
    (X_te_eye, y_te_eye),
    (X_te_apt, y_te_apt),
]

print("Client data summary (domain_id = client index):")
for i, (cx, cy) in enumerate(client_data):
    print(f"  Client {i} ({CLIENT_NAMES[i]:<8}) domain_id={i} | train: {len(cy):>6,}")

In [ ]:
# =========================
# CELL 8 — CLASS WEIGHTS PER CLIENT
# =========================
AUTOTUNE = tf.data.AUTOTUNE

def compute_client_class_weights(y, num_classes=NUM_CLASSES):
    classes  = np.unique(y)
    cw_array = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    cw = {int(c): float(w) for c, w in zip(classes, cw_array)}
    for c in range(num_classes):
        if c not in cw:
            cw[c] = 1.0
    return cw

client_class_weights = []
for i, (cx, cy) in enumerate(client_data):
    cw = compute_client_class_weights(cy)
    client_class_weights.append(cw)
    print(f"Client {i} ({CLIENT_NAMES[i]}) weights: {cw}")

In [ ]:
# =========================
# CELL 9 — TF DATASET HELPERS (DANN — dual output)
# =========================
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.05),
], name="light_augmentation")

def make_dann_client_dataset(X, y_label, domain_id, augment=True):
    """Dataset dengan dual output: label_output + domain_output."""
    y_domain = np.full(len(y_label), domain_id, dtype=np.int32)

    ds = tf.data.Dataset.from_tensor_slices((
        X,
        {"label_output": y_label.astype(np.int32),
         "domain_output": y_domain}
    ))
    ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if augment:
        def aug_fn(x, y):
            x = data_augmentation(x, training=True)
            return x, y
        ds = ds.map(aug_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

def make_eval_dataset(X, y):
    """Eval dataset — single label output (tidak butuh domain)."""
    return (
        tf.data.Dataset.from_tensor_slices((X, y))
        .batch(BATCH_SIZE).prefetch(AUTOTUNE)
    )

# Global val & test ds (eval — label only)
val_ds_global  = make_eval_dataset(X_val_global,  y_val_global)
test_ds_global = make_eval_dataset(X_test_global, y_test_global)

# Per-client DANN train ds
client_dann_ds = [
    make_dann_client_dataset(cx, cy, domain_id=i)
    for i, (cx, cy) in enumerate(client_data)
]

# Per-client eval ds
client_test_ds = [make_eval_dataset(x, y) for x, y in client_test_data]

print("All datasets ready.")
print(f"  Global val  batches : {len(val_ds_global)}")
print(f"  Global test batches : {len(test_ds_global)}")

In [ ]:
# =========================
# CELL 10 — PROGRESSIVE LAMBDA UTILITY
# =========================
def get_progressive_lambda(current_round, total_rounds, gamma=LAMBDA_GAMMA):
    """
    Progressive lambda — DANN original (Ganin et al., 2016).
    lambda(p) = 2 / (1 + exp(-gamma * p)) - 1
    p = current_round / total_rounds  (0 → 1)
    """
    p = current_round / total_rounds
    return float(2.0 / (1.0 + np.exp(-gamma * p)) - 1.0)

# Preview lambda schedule
print("Progressive Lambda Schedule:")
print(f"  {'Round':>6} | {'p':>6} | {'lambda':>8}")
print("  " + "-"*28)
for r in [1, 3, 5, 8, 10, 12, 15]:
    if r <= FL_ROUNDS:
        lam = get_progressive_lambda(r, FL_ROUNDS)
        p   = r / FL_ROUNDS
        print(f"  {r:>6} | {p:>6.2f} | {lam:>8.4f}")

In [ ]:
# =========================
# CELL 11 — GRADIENT REVERSAL LAYER
# =========================
@tf.custom_gradient
def gradient_reversal_op(x, lambda_val):
    def grad(dy):
        return -lambda_val * dy, None
    return x, grad

class GradientReversalLayer(layers.Layer):
    def __init__(self, lambda_=1.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_ = lambda_

    def call(self, x):
        lambda_tensor = tf.constant(self.lambda_, dtype=tf.float32)
        return gradient_reversal_op(x, lambda_tensor)

    def get_config(self):
        config = super().get_config()
        config.update({"lambda_": self.lambda_})
        return config

print("GradientReversalLayer ready.")

In [ ]:
# =========================
# CELL 12 — BUILD DANN MODEL COMPONENTS
# =========================
def build_feature_extractor(input_shape=(224, 224, 3), unfreeze_last_n=UNFREEZE_LAST_N_LAYERS):
    backbone = MobileNetV2(
        input_shape=input_shape, include_top=False, weights="imagenet"
    )
    backbone.trainable = False  # Stage 1: frozen

    inputs   = keras.Input(shape=input_shape, name="image_input")
    x        = backbone(inputs, training=False)
    x        = layers.GlobalAveragePooling2D()(x)
    x        = layers.BatchNormalization()(x)
    x        = layers.Dropout(0.35)(x)
    features = layers.Dense(
        FEATURE_DIM, activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="shared_features"
    )(x)
    features = layers.BatchNormalization()(features)
    features = layers.Dropout(0.30)(features)

    return keras.Model(inputs, features, name="feature_extractor"), backbone

def build_label_classifier(num_classes=NUM_CLASSES):
    inputs  = keras.Input(shape=(FEATURE_DIM,), name="label_input")
    x       = layers.Dense(64, activation="relu")(inputs)
    x       = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="label_output")(x)
    return keras.Model(inputs, outputs, name="label_classifier")

def build_domain_classifier(num_domains=NUM_DOMAINS, lambda_grl=1.0):
    inputs  = keras.Input(shape=(FEATURE_DIM,), name="domain_input")
    x       = GradientReversalLayer(lambda_=lambda_grl, name="grl")(inputs)
    x       = layers.Dense(64, activation="relu")(x)
    x       = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_domains, activation="softmax", name="domain_output")(x)
    return keras.Model(inputs, outputs, name="domain_classifier")

print("Model component builders ready.")

In [ ]:
# =========================
# CELL 13 — DANN MODEL CLASS
# =========================
class DANNModel(keras.Model):
    def __init__(self, feature_extractor, label_classifier, domain_classifier,
                 domain_loss_weight=DOMAIN_LOSS_WEIGHT, **kwargs):
        super().__init__(**kwargs)
        self.feature_extractor   = feature_extractor
        self.label_classifier    = label_classifier
        self.domain_classifier   = domain_classifier
        self.domain_loss_weight  = domain_loss_weight

        self.label_loss_fn  = keras.losses.SparseCategoricalCrossentropy()
        self.domain_loss_fn = keras.losses.SparseCategoricalCrossentropy()

        self.total_loss_tracker  = keras.metrics.Mean(name="loss")
        self.label_loss_tracker  = keras.metrics.Mean(name="label_loss")
        self.domain_loss_tracker = keras.metrics.Mean(name="domain_loss")
        self.label_acc           = keras.metrics.SparseCategoricalAccuracy(name="label_accuracy")
        self.domain_acc          = keras.metrics.SparseCategoricalAccuracy(name="domain_accuracy")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker, self.label_loss_tracker,
            self.domain_loss_tracker, self.label_acc, self.domain_acc
        ]

    def call(self, inputs, training=False):
        features     = self.feature_extractor(inputs, training=training)
        label_preds  = self.label_classifier(features,  training=training)
        domain_preds = self.domain_classifier(features, training=training)
        return {"label_output": label_preds, "domain_output": domain_preds}

    def train_step(self, data):
        x, y          = data
        y_label       = y["label_output"]
        y_domain      = y["domain_output"]

        with tf.GradientTape() as tape:
            features     = self.feature_extractor(x, training=True)
            label_preds  = self.label_classifier(features,  training=True)
            domain_preds = self.domain_classifier(features, training=True)

            label_loss   = self.label_loss_fn(y_label,  label_preds)
            domain_loss  = self.domain_loss_fn(y_domain, domain_preds)
            total_loss   = label_loss + self.domain_loss_weight * domain_loss

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        self.total_loss_tracker.update_state(total_loss)
        self.label_loss_tracker.update_state(label_loss)
        self.domain_loss_tracker.update_state(domain_loss)
        self.label_acc.update_state(y_label,  label_preds)
        self.domain_acc.update_state(y_domain, domain_preds)

        return {m.name: m.result() for m in self.metrics}

    def predict_labels(self, dataset, verbose=0):
        """Predict hanya label output (untuk eval)."""
        all_probs = []
        for batch_x, _ in dataset:
            features    = self.feature_extractor(batch_x, training=False)
            label_probs = self.label_classifier(features,  training=False)
            all_probs.append(label_probs.numpy())
        return np.concatenate(all_probs, axis=0)

print("DANNModel class ready.")

In [ ]:
# =========================
# CELL 14 — BUILD GLOBAL DANN MODEL + HELPERS
# =========================
def build_dann_model(lambda_grl=1.0, stage=1):
    feat_extractor, backbone = build_feature_extractor()

    if stage == 2:
        backbone.trainable = True
        for layer in backbone.layers[:-UNFREEZE_LAST_N_LAYERS]:
            layer.trainable = False
        for layer in backbone.layers[-UNFREEZE_LAST_N_LAYERS:]:
            layer.trainable = True
        for layer in backbone.layers:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    label_cls  = build_label_classifier()
    domain_cls = build_domain_classifier(lambda_grl=lambda_grl)

    model = DANNModel(
        feature_extractor  = feat_extractor,
        label_classifier   = label_cls,
        domain_classifier  = domain_cls,
        domain_loss_weight = DOMAIN_LOSS_WEIGHT,
        name               = "dann_model"
    )
    # Build variables
    dummy = tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3), dtype=tf.float32)
    _     = model(dummy, training=False)
    return model

def get_weights(model): return model.get_weights()
def set_weights(model, w): model.set_weights(w)

def fedavg_dann(global_w, client_w_list, client_sizes):
    total    = sum(client_sizes)
    averaged = []
    for layer_idx in range(len(global_w)):
        layer_avg = np.sum(
            [client_w_list[i][layer_idx] * client_sizes[i] / total
             for i in range(len(client_w_list))],
            axis=0
        )
        averaged.append(layer_avg)
    return averaged

# Init global model
global_model   = build_dann_model(lambda_grl=1.0, stage=1)
global_weights = get_weights(global_model)

print("Global DANN model initialized.")
global_model.summary()

In [ ]:
# =========================
# CELL 15 — LOCAL TRAINING UTILITY
# =========================
def local_train_dann(global_weights, client_X, client_y, domain_id,
                     client_cw, stage, epochs, lr, lambda_grl):
    """Build local DANN model, load global weights, train, return updated weights."""
    local_model = build_dann_model(lambda_grl=lambda_grl, stage=stage)
    local_model.set_weights(global_weights)

    local_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr)
    )

    local_ds = make_dann_client_dataset(client_X, client_y, domain_id=domain_id, augment=True)
    local_model.fit(local_ds, epochs=epochs, class_weight=client_cw, verbose=0)

    return local_model.get_weights()

print("Local DANN training utility ready.")

In [ ]:
# =========================
# CELL 16 — FL+DANN TRAINING LOOP (DANN + SIZE ASCENDING)
# =========================
fl_round_logs  = []
best_val_acc   = -1.0
best_weights   = None

print("=" * 70)
print(f"  FL+DANN — MultiClient — {SORT_STRATEGY.upper()}")
print(f"  Rounds: {FL_ROUNDS} | Clients: {CLIENT_NAMES} | Batch: {BATCH_SIZE}")
print(f"  Sort  : fixed — APTOS → DDR → EyePACS (smallest → largest)")
print(f"  Lambda: progressive (gamma={LAMBDA_GAMMA})")
print("=" * 70)

round_pbar = tqdm(range(1, FL_ROUNDS + 1), desc="FL Rounds", unit="round", ncols=85, colour="cyan")

for fl_round in round_pbar:
    round_start = time.time()

    # ── Stage & LR ──
    if fl_round <= FL_ROUNDS // 2:
        stage = 1; epochs_local = EPOCHS_LOCAL_STAGE1; lr_local = LEARNING_RATE_STAGE1
    else:
        stage = 2; epochs_local = EPOCHS_LOCAL_STAGE2; lr_local = LEARNING_RATE_STAGE2

    # ── Progressive lambda ──
    lambda_grl = get_progressive_lambda(fl_round, FL_ROUNDS)

    # ── Update GRL lambda di global model ──
    set_weights(global_model, global_weights)
    global_model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr_local))

    tqdm.write(f"\n{chr(9472)*70}")
    tqdm.write(
        f"[Round {fl_round:02d}/{FL_ROUNDS}] Stage {stage} | "
        f"LR: {lr_local} | lambda_grl: {lambda_grl:.4f}"
    )

    # ── Fixed order (size-based) ──
    sorted_order = SORTED_ORDER_FIXED
    
    tqdm.write(f"  Fixed order: {[CLIENT_NAMES[i] for i in sorted_order]}")

    # ── Local training per client ──
    client_updated_weights = []
    client_sizes           = []
    client_round_metrics   = []

    client_pbar = tqdm(
        enumerate(sorted_order), total=NUM_CLIENTS,
        desc="  Training", unit="client", ncols=70, leave=False, colour="green"
    )

    for rank, client_idx in client_pbar:
        cx, cy = client_data[client_idx]
        cw     = client_class_weights[client_idx]

        client_pbar.set_description(f"  {CLIENT_NAMES[client_idx]} [{rank+1}/{NUM_CLIENTS}]")

        t0        = time.time()
        updated_w = local_train_dann(
            global_weights = global_weights,
            client_X       = cx,
            client_y       = cy,
            domain_id      = client_idx,
            client_cw      = cw,
            stage          = stage,
            epochs         = epochs_local,
            lr             = lr_local,
            lambda_grl     = lambda_grl
        )
        elapsed = time.time() - t0

        # Quick eval post-train (label loss only)
        tmp_model = build_dann_model(lambda_grl=lambda_grl, stage=stage)
        tmp_model.set_weights(updated_w)
        c_label_loss = evaluate_client_label_loss(tmp_model, cx, cy)
        del tmp_model

        client_updated_weights.append(updated_w)
        client_sizes.append(len(cx))

        metric_entry = {
            "client_idx"       : int(client_idx),
            "client_name"      : CLIENT_NAMES[client_idx],
            "rank_in_round"    : rank + 1,
            "num_samples"      : int(len(cx)),
            "label_loss_after" : float(c_label_loss),
            "lambda_grl"       : float(lambda_grl),
            "train_time_sec"   : round(elapsed, 2)
        }
        
        client_round_metrics.append(metric_entry)

        tqdm.write(
            f"  ✓ {CLIENT_NAMES[client_idx]} (rank {rank+1}) | "
            f"label_loss: {c_label_loss:.4f} | λ: {lambda_grl:.4f} | {elapsed:.1f}s"
        )

    # ── FedAvg ──
    global_weights = fedavg_dann(global_weights, client_updated_weights, client_sizes)
    set_weights(global_model, global_weights)

    # ── Global val eval (label only) ──
    val_probs    = global_model.predict_labels(val_ds_global, verbose=0)
    val_preds    = np.argmax(val_probs, axis=1)
    val_acc      = float(accuracy_score(y_val_global, val_preds))
    val_loss_fn  = keras.losses.SparseCategoricalCrossentropy()
    val_loss_val = float(tf.reduce_mean(val_loss_fn(y_val_global.astype(np.int32),
                                                      val_probs.astype(np.float32))).numpy())

    round_elapsed = time.time() - round_start

    round_pbar.set_postfix(
        val_acc=f"{val_acc:.4f}", val_loss=f"{val_loss_val:.4f}",
        lambda_=f"{lambda_grl:.3f}", time=f"{round_elapsed:.0f}s"
    )
    tqdm.write(
        f"\n  ▶ [Round {fl_round:02d}] Val Acc: {val_acc:.4f} | "
        f"Val Loss: {val_loss_val:.4f} | λ: {lambda_grl:.4f} | {round_elapsed:.1f}s"
    )

    # ── Track best ──
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = [w.copy() for w in global_weights]
        tqdm.write(f"  ★ New best val_acc: {best_val_acc:.4f}")

    round_log = {
        "round"          : fl_round,
        "stage"          : stage,
        "strategy"       : SORT_STRATEGY,
        "lambda_grl"     : float(lambda_grl),
        "sorted_order"   : [CLIENT_NAMES[i] for i in sorted_order],
        "val_loss"       : float(val_loss_val),
        "val_accuracy"   : float(val_acc),
        "round_time_sec" : round(round_elapsed, 2),
        "client_metrics" : client_round_metrics
    }
    
    fl_round_logs.append(round_log)

    global_model.save_weights(os.path.join(MODELS_DIR, f"global_dann_round_{fl_round:02d}.weights.h5"))

tqdm.write("\n" + "=" * 70)
tqdm.write(f"  FL+DANN TRAINING COMPLETE | Best Val Acc: {best_val_acc:.4f}")
tqdm.write("=" * 70)

# Load best weights untuk evaluasi final
if best_weights is not None:
    set_weights(global_model, best_weights)
    print(f"Loaded best weights (val_acc={best_val_acc:.4f}) for final evaluation.")

In [ ]:
# =========================
# CELL 17 — SAVE FL ROUND LOGS
# =========================
fl_logs_path = os.path.join(LOGS_DIR, "fl_round_logs.json")
with open(fl_logs_path, "w") as f:
    json.dump(fl_round_logs, f, indent=4)

rounds_summary = pd.DataFrame([{
    "round"       : r["round"],
    "stage"       : r["stage"],
    "lambda_grl"  : r["lambda_grl"],
    "sorted_order": str(r["sorted_order"]),
    "val_loss"    : r["val_loss"],
    "val_accuracy": r["val_accuracy"],
    "round_time_s": r["round_time_sec"],
} for r in fl_round_logs])
display(rounds_summary)

rounds_summary.to_csv(os.path.join(LOGS_DIR, "fl_round_summary.csv"), index=False)
print("Saved:", fl_logs_path)

In [ ]:
# =========================
# CELL 18 — SAVE FINAL MODEL
# =========================
FINAL_WEIGHTS_PATH = os.path.join(MODELS_DIR, "global_dann_best.weights.h5")
global_model.save_weights(FINAL_WEIGHTS_PATH)
print("Best model weights saved to:", FINAL_WEIGHTS_PATH)

In [ ]:
# =========================
# CELL 19 — PLOT LAMBDA SCHEDULE + TRAINING CURVES
# =========================
rounds_list    = [r["round"]        for r in fl_round_logs]
val_acc_list   = [r["val_accuracy"] for r in fl_round_logs]
val_loss_list  = [r["val_loss"]     for r in fl_round_logs]
lambda_list    = [r["lambda_grl"]   for r in fl_round_logs]
stage2_start   = next((r["round"] for r in fl_round_logs if r["stage"] == 2), None)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Accuracy
axes[0].plot(rounds_list, val_acc_list, marker="o", label="Val Accuracy")
if stage2_start:
    axes[0].axvline(x=stage2_start, color="gray", linestyle="--", linewidth=1, label="Stage 1→2")
axes[0].set_title("Val Accuracy per Round\nDANN + Size Ascending")
axes[0].set_xlabel("FL Round"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True)

# Loss
axes[1].plot(rounds_list, val_loss_list, marker="o", color="orange", label="Val Loss")
if stage2_start:
    axes[1].axvline(x=stage2_start, color="gray", linestyle="--", linewidth=1, label="Stage 1→2")
axes[1].set_title("Val Loss per Round\nDANN + Size Ascending")
axes[1].set_xlabel("FL Round"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True)

# Lambda schedule
axes[2].plot(rounds_list, lambda_list, marker="s", color="green", label="Lambda GRL")
if stage2_start:
    axes[2].axvline(x=stage2_start, color="gray", linestyle="--", linewidth=1, label="Stage 1→2")
axes[2].set_title("Progressive Lambda Schedule\nDANN GRL")
axes[2].set_xlabel("FL Round"); axes[2].set_ylabel("Lambda")
axes[2].legend(); axes[2].grid(True)

plt.suptitle(f"MultiClient FL+DANN — {SORT_STRATEGY} | DDR + EyePACS + APTOS", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "fl_dann_training_curves.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: fl_dann_training_curves.png")

In [ ]:
# =========================
# CELL 20 — BUILD FINAL PREDICTION OBJECTS
# =========================
test_probs = global_model.predict_labels(test_ds_global, verbose=1)
test_preds = np.argmax(test_probs, axis=1)

y_true = np.array(y_test_global)
y_pred = np.array(test_preds)
y_prob = np.array(test_probs)

print("y_true:", y_true.shape, "| y_pred:", y_pred.shape, "| y_prob:", y_prob.shape)
print("Unique labels:", np.unique(y_true))

In [ ]:
# =========================
# CELL 21 — MAIN SUMMARY TABLE
# =========================
acc = accuracy_score(y_true, y_pred)

precision_macro,    recall_macro,    f1_macro,    _ = precision_recall_fscore_support(y_true, y_pred, average="macro",    zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

y_true_bin       = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
auc_macro_ovr    = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="macro")
auc_weighted_ovr = roc_auc_score(y_true_bin, y_prob, multi_class="ovr", average="weighted")

summary_df = pd.DataFrame([{
    "experiment_name"   : EXPERIMENT_NAME,
    "dataset_name"      : DATASET_NAME,
    "model_name"        : MODEL_NAME,
    "setup"             : f"FL+DANN {SORT_STRATEGY}",
    "evaluation_set"    : "Test (Global — Best Val Weights)",
    "accuracy"          : acc,
    "precision_macro"   : precision_macro,
    "recall_macro"      : recall_macro,
    "f1_macro"          : f1_macro,
    "precision_weighted": precision_weighted,
    "recall_weighted"   : recall_weighted,
    "f1_weighted"       : f1_weighted,
    "auc_macro_ovr"     : auc_macro_ovr,
    "auc_weighted_ovr"  : auc_weighted_ovr,
    "fl_rounds"         : FL_ROUNDS,
    "best_val_acc"      : best_val_acc,
    "lambda_gamma"      : LAMBDA_GAMMA,
    "domain_loss_weight": DOMAIN_LOSS_WEIGHT,
    "num_clients"       : NUM_CLIENTS,
    "total_train_size"  : sum(len(cy) for _, cy in client_data),
    "global_test_size"  : len(y_test_global),
}])

display(summary_df)
summary_df.to_csv( os.path.join(LOGS_DIR, "baseline_summary_table.csv"),  index=False)
summary_df.to_json(os.path.join(LOGS_DIR, "baseline_summary_table.json"), orient="records", indent=4)
print("Saved: baseline_summary_table.csv / .json")

In [ ]:
# =========================
# CELL 22 — CLASSIFICATION REPORT
# =========================
report_text = classification_report(y_true, y_pred, digits=4)
report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)
print(report_text)

with open(os.path.join(LOGS_DIR, "classification_report_test.txt"),  "w") as f: f.write(report_text)
with open(os.path.join(LOGS_DIR, "classification_report_test.json"), "w") as f: json.dump(report_dict, f, indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 23 — PER-CLASS METRICS
# =========================
precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
    y_true, y_pred, labels=np.arange(NUM_CLASSES), zero_division=0
)
per_class_df = pd.DataFrame({
    "class"    : np.arange(NUM_CLASSES),
    "precision": precision_cls,
    "recall"   : recall_cls,
    "f1_score" : f1_cls,
    "support"  : support_cls
})
display(per_class_df)
per_class_df.to_csv( os.path.join(LOGS_DIR, "per_class_metrics.csv"),  index=False)
per_class_df.to_json(os.path.join(LOGS_DIR, "per_class_metrics.json"), orient="records", indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 24 — ROC CURVE + AUC
# =========================
y_true_bin    = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
roc_auc_dict  = {}

plt.figure(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    auc_val     = auc(fpr, tpr)
    roc_auc_dict[f"class_{i}"] = float(auc_val)
    plt.plot(fpr, tpr, label=f"Class {i} (AUC={auc_val:.4f})")

plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title(f"ROC Curve — FL+DANN DANN + Size Ascending\nDDR + EyePACS + APTOS")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "roc_curve_baseline.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

with open(os.path.join(LOGS_DIR, "roc_auc_per_class.json"), "w") as f:
    json.dump(roc_auc_dict, f, indent=4)
print("Saved: roc_curve_baseline.png + roc_auc_per_class.json")

In [ ]:
# =========================
# CELL 25 — CONFUSION MATRIX
# =========================
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(NUM_CLASSES))
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix — FL+DANN DANN + Size Ascending")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "confusion_matrix_heatmap_baseline.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()

cm_df = pd.DataFrame(cm,
    index=[f"true_{i}" for i in range(NUM_CLASSES)],
    columns=[f"pred_{i}" for i in range(NUM_CLASSES)])
cm_df.to_csv(os.path.join(LOGS_DIR, "confusion_matrix_table.csv"))
with open(os.path.join(LOGS_DIR, "confusion_matrix_values_test.json"), "w") as f:
    json.dump({"confusion_matrix": cm.tolist()}, f, indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 26 — PER-CLIENT FINAL EVALUATION
# =========================
print("Per-Client Performance on Test Set (Best Global DANN Model)")
print("=" * 65)

per_client_results = []

for i, (name, (X_te, y_te)) in enumerate(zip(CLIENT_NAMES, client_test_data)):
    te_ds   = make_eval_dataset(X_te, y_te)
    probs_i = global_model.predict_labels(te_ds, verbose=0)
    preds_i = np.argmax(probs_i, axis=1)
    y_bin_i = label_binarize(y_te, classes=np.arange(NUM_CLASSES))

    c_acc           = accuracy_score(y_te, preds_i)
    c_p, c_r, c_f1, _ = precision_recall_fscore_support(y_te, preds_i, average="macro", zero_division=0)
    try:
        c_auc = roc_auc_score(y_bin_i, probs_i, multi_class="ovr", average="macro")
    except ValueError:
        c_auc = None

    auc_str = f"{c_auc:.4f}" if c_auc is not None else "N/A"

    per_client_results.append({
        "client"         : name,
        "n_test_samples" : int(len(y_te)),
        "accuracy"       : round(c_acc, 4),
        "precision_macro": round(c_p,   4),
        "recall_macro"   : round(c_r,   4),
        "f1_macro"       : round(c_f1,  4),
        "auc_macro_ovr"  : round(c_auc, 4) if c_auc is not None else None,
    })
    print(f"  {name:<10} | n={len(y_te):>5,} | Acc={c_acc:.4f} | F1={c_f1:.4f} | AUC={auc_str}")

print()
per_client_df = pd.DataFrame(per_client_results)
display(per_client_df)
per_client_df.to_csv( os.path.join(LOGS_DIR, "per_client_test_metrics.csv"),  index=False)
per_client_df.to_json(os.path.join(LOGS_DIR, "per_client_test_metrics.json"), orient="records", indent=4)
print("Saved.")

In [ ]:
# =========================
# CELL 27 — DOMAIN ACCURACY ANALYSIS (DANN Evidence)
# =========================
# Analisis seberapa baik domain classifier bisa/tidak bisa bedain asal dataset
# Kalau domain accuracy RENDAH = DANN berhasil buat domain-invariant features

print("Domain Classifier Analysis per Round")
print("=" * 55)
print("(Low domain acc = DANN successfully confused domain classifier)")
print()

domain_acc_per_round = []
for r in fl_round_logs:
    for cm_entry in r["client_metrics"]:
        domain_acc_per_round.append({
            "round"      : r["round"],
            "client"     : cm_entry["client_name"],
            "lambda_grl" : r["lambda_grl"],
            "val_accuracy": r["val_accuracy"],
        })

domain_df = pd.DataFrame(domain_acc_per_round)

# Plot val accuracy vs lambda progress
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Val acc vs round (colored by stage)
colors = ["#2196F3" if r["stage"] == 1 else "#FF9800" for r in fl_round_logs]
axes[0].bar([r["round"] for r in fl_round_logs],
            [r["val_accuracy"] for r in fl_round_logs],
            color=colors, alpha=0.8, label="Val Accuracy")
axes[0].set_title("Val Accuracy per Round (Blue=Stage1, Orange=Stage2)")
axes[0].set_xlabel("Round"); axes[0].set_ylabel("Accuracy")
axes[0].grid(axis="y")

# Lambda vs round
axes[1].plot([r["round"] for r in fl_round_logs],
             [r["lambda_grl"] for r in fl_round_logs],
             marker="o", color="green")
axes[1].set_title("Progressive Lambda (GRL Strength) per Round")
axes[1].set_xlabel("Round"); axes[1].set_ylabel("Lambda GRL")
axes[1].grid(True)

plt.suptitle(f"DANN Domain Analysis — {SORT_STRATEGY}", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "dann_domain_analysis.png"), dpi=300, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: dann_domain_analysis.png")

In [ ]:
# =========================
# CELL 28 — COMPARISON READY JSON
# =========================
comparison_row = {
    "method"            : f"FL_DANN_{SORT_STRATEGY}",
    "model"             : MODEL_NAME,
    "dataset"           : DATASET_NAME,
    "clients"           : CLIENT_NAMES,
    "accuracy"          : float(acc),
    "precision_macro"   : float(precision_macro),
    "recall_macro"      : float(recall_macro),
    "f1_macro"          : float(f1_macro),
    "auc_macro_ovr"     : float(auc_macro_ovr),
    "fl_rounds"         : FL_ROUNDS,
    "best_val_acc"      : float(best_val_acc),
    "lambda_gamma"      : LAMBDA_GAMMA,
    "domain_loss_weight": DOMAIN_LOSS_WEIGHT,
    "num_clients"       : NUM_CLIENTS,
    "total_train_size"  : int(sum(len(cy) for _, cy in client_data)),
    "global_test_size"  : int(len(y_test_global)),
    "notes"             : (
        f"FL+DANN {SORT_STRATEGY}, progressive lambda (gamma={LAMBDA_GAMMA}), "
        f"{FL_ROUNDS} rounds, {NUM_CLIENTS} clients. "
        f"Best weights from round with highest val_acc={best_val_acc:.4f}."
    )
}

path = os.path.join(LOGS_DIR, "comparison_ready_baseline.json")
with open(path, "w") as f:
    json.dump(comparison_row, f, indent=4)
print("Saved:", path)
print(json.dumps(comparison_row, indent=4))